# Обучение многообразий: визуализация высокоразмерных данных методами t-SNE и UMAP

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Обучение многообразий (UMAP и t-SNE)».

## Что делает метод

Когда данные описываются сотнями признаков — геномный профиль клетки, масс-спектр вещества, анкета с сотнями пунктов — ни таблица, ни обычный график не дают интуитивного понимания структуры. Методы обучения многообразий (manifold learning) решают эту задачу: они сворачивают многомерное «облако» точек в двумерную карту так, чтобы похожие наблюдения оказались рядом.

На такой карте сразу видно, распадаются ли данные на группы, есть ли плавные переходы между состояниями, где расположены нетипичные объекты. Два наиболее распространённых метода — **t-SNE** и **UMAP** — различаются скоростью и тем, насколько хорошо сохраняют глобальное расположение групп, но оба предназначены прежде всего для **визуализации**, а не для предсказания.

В этом блокноте мы сравним PCA, t-SNE и UMAP на одном наборе данных, разберём влияние ключевых гиперпараметров и укажем, что именно можно и нельзя читать с получаемых карт. Расчётное время прохождения — около семи минут.

## Интуиция за методом

Представьте, что вы разворачиваете смятый лист бумаги, на котором нарисована карта: лист трёхмерный, но в нём скрыта плоская двумерная структура. Алгоритм находит эту структуру, разворачивает её и кладёт перед вами — ценой небольших искажений в отдельных местах, но с сохранением главного: кто кому сосед. Скрытая структура низкой размерности, на которой лежат данные, и называется **многообразием** (manifold).

В отличие от PCA, который ищет линейные оси максимального разброса, t-SNE и UMAP строят **граф соседства** — кто с кем близок в исходном многомерном пространстве — и затем располагают точки на плоскости так, чтобы этот граф сохранился.

**Ключевые понятия, которые встретятся в блокноте:**

- **Многообразие (manifold)** — гладкая низкоразмерная структура, скрытая внутри высокоразмерного пространства признаков (пример: траектория развития клетки в пространстве из 20 000 генов).
- **Граф соседства** — сеть, в которой каждая точка соединена с ближайшими соседями в исходном пространстве; именно этот граф оба метода стараются сохранить в проекции.
- **Perplexity (t-SNE)** — гиперпараметр, задающий эффективное число соседей; управляет балансом между локальной детализацией и глобальной структурой.
- **n\_neighbors (UMAP)** — число ближайших соседей при построении графа; аналог perplexity: малые значения акцентируют локальные детали, большие сохраняют глобальную структуру.
- **min\_dist (UMAP)** — минимальное расстояние между точками в проекции; малые значения дают плотные кластеры, большие — равномерное распределение.
- **PCA (метод главных компонент)** — линейный метод снижения размерности, используемый здесь как базовая линия и как предобработка перед t-SNE/UMAP.
- **random\_state** — фиксированное зерно генератора случайных чисел; без него результат меняется от запуска к запуску.

## 1. Установка библиотек

В среде Google Colab большинство библиотек уже предустановлено. Ячейка ниже гарантирует нужные версии, в том числе при локальном запуске. `umap-learn` устанавливается отдельно — это не входит в стандартный дистрибутив scikit-learn.

In [ ]:
%pip install -q scikit-learn umap-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем набор рукописных цифр `digits` из `scikit-learn`: 1797 изображений цифр от 0 до 9, каждое описывается 64 признаками (пиксели сетки 8×8). Истинные метки цифр нам известны, но в задаче обучения многообразий мы их не используем — они нужны только для окраски точек на карте и оценки визуального качества разделения.

Данный набор — типичный аналог геномных или спектральных данных: большое число признаков, визуально неразличимые наблюдения в «сыром» виде, но заведомо известная структура из 10 классов, которую должны раскрыть методы снижения размерности.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# Загружаем набор рукописных цифр: 1797 наблюдений, 64 признака.
digits = load_digits()
X_raw = digits.data          # матрица признаков (пиксели)
y = digits.target            # истинные метки цифр (0–9), только для визуализации
n_classes = len(np.unique(y))

# Стандартизация обязательна: t-SNE и UMAP чувствительны к масштабу признаков.
X = StandardScaler().fit_transform(X_raw)

print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}, классов: {n_classes}")
print(f"Пример первых 5 меток: {y[:5]}")

## 4. Применение метода

### Шаг 4.1. PCA как предобработка и базовая линия

При высокой размерности (64 признака) рекомендуется сначала снизить её до 30–50 компонент методом PCA. Это убирает шум и ускоряет последующие алгоритмы, сохраняя основную структуру данных.

PCA также выступает **базовой линией**: сравнение его двумерной проекции с t-SNE и UMAP покажет, какую дополнительную структуру раскрывают нелинейные методы.

In [ ]:
from sklearn.decomposition import PCA

# Предварительное снижение до 30 компонент — стандартная практика
# перед запуском t-SNE и UMAP на данных с высокой размерностью.
pca_pre = PCA(n_components=30, random_state=42)
X_pca30 = pca_pre.fit_transform(X)

# Двумерная проекция PCA — базовая линия для сравнения.
pca_2d = PCA(n_components=2, random_state=42)
X_pca2 = pca_2d.fit_transform(X)

explained = pca_2d.explained_variance_ratio_.sum()
print(f"PCA 30 компонент объясняет {pca_pre.explained_variance_ratio_.sum():.1%} дисперсии.")
print(f"PCA 2 компоненты объясняют {explained:.1%} дисперсии — значительная часть структуры теряется.")

### Шаг 4.2. t-SNE

**t-SNE** преобразует расстояния в вероятности: пары близких точек получают высокую вероятность быть «соседями». В проекции используется распределение Стьюдента (отсюда «t»), которое лучше разделяет кластеры в 2D. Алгоритм минимизирует расхождение Кульбака–Лейблера между распределениями в исходном и проекционном пространствах.

Параметр `perplexity` (здесь 30) задаёт эффективное число соседей; `random_state` фиксирует результат.

In [ ]:
from sklearn.manifold import TSNE

# t-SNE запускается на предобработанных PCA данных (30 компонент).
# perplexity: рекомендуемый диапазон 5–50; здесь 30 — хороший выбор по умолчанию.
# max_iter: число итераций оптимизации; 1000 достаточно для данного набора.
tsne = TSNE(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    init="pca",        # инициализация через PCA стабилизирует результат
)
X_tsne = tsne.fit_transform(X_pca30)
print(f"t-SNE завершён. Форма результата: {X_tsne.shape}")

### Шаг 4.3. UMAP

**UMAP** строит взвешенный граф k ближайших соседей в исходном пространстве, а затем оптимизирует расположение точек в 2D, минимизируя бинарную кросс-энтропию между исходным и проекционным графами. UMAP значительно быстрее t-SNE и лучше сохраняет глобальную структуру данных.

In [ ]:
import umap

# UMAP запускается на тех же 30-компонентных данных, что и t-SNE.
# n_neighbors: число ближайших соседей (аналог perplexity); рекомендуется 5–50.
# min_dist: минимальное расстояние в проекции; 0.1 — хорошее значение по умолчанию.
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)
X_umap = reducer.fit_transform(X_pca30)
print(f"UMAP завершён. Форма результата: {X_umap.shape}")

### Шаг 4.4. Сравнение PCA, t-SNE и UMAP на одном рисунке

Ячейка ниже строит три карты бок о бок. Каждая точка — одно изображение цифры; цвет — её истинный класс (0–9). Обратите внимание, насколько по-разному три метода раскрывают структуру одних и тех же данных.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Расширенная палитра для 10 классов: циклически из токенов темы + дополнительные цвета.
CLASS_COLORS = [
    "#2563eb", "#0d9488", "#b45309", "#4d5e78",
    "#7c3aed", "#db2777", "#059669", "#d97706",
    "#1d4ed8", "#065f46",
]

projections = [
    (X_pca2,  "PCA (линейный метод, базовая линия)"),
    (X_tsne,  "t-SNE  (perplexity=30)"),
    (X_umap,  "UMAP  (n_neighbors=15, min_dist=0.1)"),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for ax, (coords, title) in zip(axes, projections):
    for cls in range(n_classes):
        mask = y == cls
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            s=12, alpha=0.7,
            color=CLASS_COLORS[cls],
            label=str(cls),
        )
    ax.set_title(title)
    ax.set_xlabel("Ось 1 проекции")
    ax.set_ylabel("Ось 2 проекции")
    ax.legend(
        title="Цифра", loc="best",
        fontsize=8, title_fontsize=9,
        markerscale=1.5, ncol=2,
    )

fig.suptitle(
    "Снижение размерности: три метода на наборе digits",
    fontsize=14, fontweight="semibold", y=1.01,
)
fig.tight_layout()
plt.show()

**Как читать эти карты.**

- **PCA**: классы частично перекрываются — линейный метод не способен «вытащить» нелинейную структуру рукописных цифр. Оси главных компонент интерпретируемы как направления максимального разброса, но два измерения сохраняют лишь около 28% дисперсии.
- **t-SNE**: классы разделены значительно чётче, часто образуя компактные облака. Локальная структура соседства сохранена хорошо, однако **расстояния между кластерами и их размеры количественно не интерпретируемы** — два далёких кластера на карте могут быть не дальше от близких, чем это выглядит.
- **UMAP**: аналогичное или лучшее разделение при меньшем времени вычислений и лучшем сохранении глобальной структуры. Те же предупреждения о расстояниях применимы в полной мере.

> **Важное предупреждение.** Ни для t-SNE, ни для UMAP нельзя делать выводы о том, насколько один кластер «дальше» от другого, или о том, что большой кластер «многочисленнее» маленького. Карта показывает только локальное соседство, а не метрику исходного пространства.

## 5. Наглядный «ага»-эксперимент: чувствительность к гиперпараметрам

Один из главных рисков работы с t-SNE и UMAP — **артефактные кластеры**: структура, которая выглядит убедительно, но является следствием неудачного выбора гиперпараметров, а не свойством данных.

Ячейки ниже демонстрируют, как меняется карта t-SNE при разных значениях `perplexity` и карта UMAP при разных значениях `n_neighbors`. Правило надёжного анализа: структура, которая воспроизводится при нескольких значениях гиперпараметров, заслуживает доверия; структура, которая появляется только при одном конкретном значении — артефакт.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Три значения perplexity: малое (5), стандартное (30), большое (80).
perplexity_values = [5, 30, 80]
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for ax, perp in zip(axes, perplexity_values):
    emb = TSNE(
        n_components=2, perplexity=perp,
        max_iter=1000, random_state=42, init="pca",
    ).fit_transform(X_pca30)

    for cls in range(n_classes):
        mask = y == cls
        ax.scatter(
            emb[mask, 0], emb[mask, 1],
            s=10, alpha=0.7, color=CLASS_COLORS[cls], label=str(cls),
        )
    ax.set_title(f"t-SNE  perplexity={perp}")
    ax.set_xlabel("Ось 1")
    ax.set_ylabel("Ось 2")

# Легенду показываем только на среднем графике.
axes[1].legend(title="Цифра", fontsize=8, title_fontsize=9,
               markerscale=1.5, ncol=2, loc="best")

fig.suptitle(
    "t-SNE: влияние параметра perplexity на структуру карты",
    fontsize=14, fontweight="semibold", y=1.01,
)
fig.tight_layout()
plt.show()

**Что демонстрирует этот эксперимент.**

- При `perplexity=5` (малое число «соседей»): алгоритм видит только ближайшее окружение каждой точки. Группы могут дробиться на мелкие фрагменты, не отражающие реальных подструктур.
- При `perplexity=30` (стандартное): баланс между локальной и глобальной структурой. Классы хорошо разделены.
- При `perplexity=80` (большое): алгоритм учитывает широкое окружение, кластеры «сжимаются» и могут терять чёткость внутренней структуры.

Сравните: те классы, что разделены при всех трёх значениях, — реально различимые группы. Разделения, появляющиеся только при одном значении — кандидаты на артефакт.

In [ ]:
import umap
import matplotlib.pyplot as plt

# Три значения n_neighbors: малое (5), стандартное (15), большое (50).
neighbor_values = [5, 15, 50]
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

for ax, nn in zip(axes, neighbor_values):
    emb = umap.UMAP(
        n_components=2, n_neighbors=nn,
        min_dist=0.1, random_state=42,
    ).fit_transform(X_pca30)

    for cls in range(n_classes):
        mask = y == cls
        ax.scatter(
            emb[mask, 0], emb[mask, 1],
            s=10, alpha=0.7, color=CLASS_COLORS[cls], label=str(cls),
        )
    ax.set_title(f"UMAP  n_neighbors={nn}")
    ax.set_xlabel("Ось 1")
    ax.set_ylabel("Ось 2")

axes[1].legend(title="Цифра", fontsize=8, title_fontsize=9,
               markerscale=1.5, ncol=2, loc="best")

fig.suptitle(
    "UMAP: влияние параметра n_neighbors на структуру карты",
    fontsize=14, fontweight="semibold", y=1.01,
)
fig.tight_layout()
plt.show()

**Что демонстрирует этот эксперимент.**

- При `n_neighbors=5`: акцент на локальной структуре, кластеры более разрозненны и фрагментированы.
- При `n_neighbors=15`: хороший баланс между локальной и глобальной структурой.
- При `n_neighbors=50`: алгоритм учитывает более широкое окружение, кластеры компактнее, но часть деталей внутри групп теряется.

Как и у t-SNE, устойчивые структуры появляются при нескольких значениях параметра. Анализируйте только те разделения, которые воспроизводятся.

## 6. Интерпретация результата

### Что можно читать с карты t-SNE и UMAP

- **Наличие групп**: если точки образуют компактные облака, разделённые пустым пространством, — в данных есть локально-различимые группы.
- **Непрерывные траектории**: если точки вытянуты в дугу или ленту — данные лежат на непрерывном многообразии (например, стадии дифференцировки клеток).
- **Выбросы**: точки вдали от всех облаков — кандидаты на нетипичные наблюдения, заслуживающие проверки.
- **Смешение групп**: если облака одного цвета перемешаны с другим — возможно, эти группы действительно не различимы по признакам, либо границы условны.

### Что нельзя читать с карты

- **Расстояния между кластерами** не отражают реальные расстояния в исходном пространстве: два близких кластера на карте могут быть дальше друг от друга, чем два далёких.
- **Размеры кластеров** не соответствуют числу наблюдений или плотности в исходном пространстве.
- **Оси проекции** не имеют содержательного смысла — в отличие от главных компонент PCA.
- **Абсолютные координаты** точек на карте произвольны и меняются от запуска к запуску (при разном `random_state`).

### Как проверять надёжность

1. Запустить с несколькими значениями гиперпараметров (perplexity или n\_neighbors) и несколькими `random_state`.
2. Сравнить с результатом другого метода (t-SNE vs UMAP) и с PCA.
3. Подтвердить найденные группы формальной кластеризацией (k-means, HDBSCAN) на исходных признаках, не на проекции.

## 7. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу числовых признаков (строки — наблюдения, столбцы — измерения). Метки или группы, если они есть, используются только для окраски точек, но не для построения проекции.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите путь к файлу.
3. При необходимости укажите столбцы-идентификаторы (не признаки) и столбец с метками.
4. Последовательно выполните ячейки разделов 4 и 5.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
#
# df = pd.read_csv("my_data.csv")          # ваш файл (загрузите через панель слева)
#
# label_column = None                      # имя столбца с метками групп (или None)
# id_columns   = []                        # столбцы-идентификаторы, не признаки
#
# cols_to_drop = ([label_column] if label_column else []) + id_columns
# X_raw_my = df.drop(columns=cols_to_drop, errors="ignore")
# X_raw_my = X_raw_my.select_dtypes(include="number")  # только числовые столбцы
#
# # Стандартизация обязательна.
# X_my = StandardScaler().fit_transform(X_raw_my)
# y_my = df[label_column].values if label_column else None
#
# print(f"Наблюдений: {X_my.shape[0]}, признаков: {X_my.shape[1]}")
#
# # При более 50 признаков — предобработка через PCA.
# from sklearn.decomposition import PCA
# n_pre = min(30, X_my.shape[1] - 1)
# X_pre_my = PCA(n_components=n_pre, random_state=42).fit_transform(X_my)
#
# # Запустите t-SNE и UMAP, заменив X_pca30 на X_pre_my в ячейках раздела 4.
# # Для окраски точек замените y на y_my (или пропустите аргумент color при y_my=None).

## Поэкспериментируйте сами

На демонстрационном наборе попробуйте следующие изменения и наблюдайте за результатом:

| Что изменить | На что обратить внимание |
|---|---|
| `perplexity` в TSNE от 5 до 100 | Как дробятся и сливаются кластеры? При каком значении цифры 4 и 9 разделены наиболее чётко? |
| `min_dist=0.5` вместо `0.1` в UMAP | Кластеры становятся более «рыхлыми» и равномерно распределёнными. |
| Убрать предобработку PCA (`X_pca30` → `X`) | Как изменится время вычислений и качество проекции? |
| Зафиксировать `random_state=0` вместо `42` | Меняется ли глобальная структура карты или только расположение кластеров? |
| `n_iter=250` в t-SNE | Проекция недооптимизирована: видно ли это визуально? |

## Краткие выводы

- t-SNE и UMAP — нелинейные методы снижения размерности для **визуализации**: они раскрывают локальную структуру соседства в данных, которую линейный PCA не улавливает.
- Оба метода строят граф соседства в исходном пространстве и оптимизируют его сохранение в 2D. UMAP быстрее t-SNE и лучше сохраняет глобальную структуру; t-SNE часто даёт более чёткое визуальное разделение локальных кластеров.
- **Стандартизация признаков и предобработка через PCA** (при высокой размерности) обязательны перед обоими методами.
- **Расстояния между кластерами и их размеры на карте не несут количественного смысла.** Интерпретировать можно только факт наличия групп и их внутреннюю плотность.
- Для проверки надёжности найденных структур всегда запускайте метод с несколькими значениями гиперпараметров и несколькими `random_state`.
- Карта t-SNE/UMAP — отправная точка исследования, а не окончательный результат: найденные группы должны быть подтверждены формальной кластеризацией на исходных признаках.
- t-SNE не поддерживает проецирование новых точек без переобучения; UMAP поддерживает через `transform`, что делает его применимым в производственных пайплайнах.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На карте t-SNE два кластера разделены широким пустым пространством. Коллега делает вывод, что эти группы «очень далеко» друг от друга в исходных данных. Что неверно в этом рассуждении и как получить корректную оценку расстояния между группами?

2. Вы запустили UMAP с `n_neighbors=5` и `n_neighbors=50`: при малом значении один из ваших экспериментальных классов распадается на три подоблака, при большом — сливается в одно. Какой вывод следует сделать о биологической природе этого класса и что нужно сделать дальше?

3. Исследователь обучил UMAP на обучающей выборке и хочет спроецировать новые образцы без повторного запуска на всём наборе. Каким методом UMAP это поддерживает и почему аналогичный подход невозможен для t-SNE?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Расстояния между кластерами на карте t-SNE не отражают расстояния в исходном пространстве: алгоритм минимизирует расхождение KL между распределениями вероятностей, а не сохраняет метрику. Широкое пространство между кластерами на карте — артефакт оптимизации. Для корректной оценки следует вычислить расстояние между центроидами кластеров в исходном признаковом пространстве (например, евклидово расстояние или косинусное сходство) или использовать PCA и сравнить положение групп вдоль главных компонент.

2. Это признак неоднородности класса: при малом n\_neighbors алгоритм видит локальные подгруппы (возможно, биологически значимые субтипы), при большом — усредняет их. Ни один из результатов сам по себе не является ответом. Следующие шаги: (а) запустить формальную кластеризацию (например, HDBSCAN) на исходных признаках внутри этого класса; (б) проверить, соответствуют ли подоблака каким-либо метаданным (время сбора, условия эксперимента, донор).

3. UMAP поддерживает метод `transform`: обученный редуктор (`reducer.fit(X_train)`) хранит граф соседства и может проецировать новые точки через `reducer.transform(X_new)` без переобучения. t-SNE не имеет отдельного этапа `transform` — алгоритм оптимизирует координаты только всего набора целиком; добавление новых точек требует повторного запуска на объединённом наборе. Именно поэтому UMAP предпочтителен при необходимости проецировать новые наблюдения в продуктивной среде.
</details>